<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/exercise_3B_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise task 3B: creating a dataset from corpus data (solution)

This notebook shows an example solution for exercise 3B.


---

# Setup

In [ ]:
!pip install --quiet datasets

In [ ]:
import random

import datasets
from datasets import Dataset, DatasetDict

---

# Process data into dataset

Download raw data from given URLs to local drive

In [ ]:
!wget --quiet -nc http://dl.turkunlp.org/TKO_7095_2023/imdb-positives.txt
!wget --quiet -nc http://dl.turkunlp.org/TKO_7095_2023/imdb-negatives.txt

Read in one text per line format

In [ ]:
positive_texts = open('imdb-positives.txt').readlines()
negative_texts = open('imdb-negatives.txt').readlines()

Reformat as list of dictionaries with `text` and `label`

In [ ]:
# List comprehension
positive_examples = [{ 'text': text, 'label': 'positive' } for text in positive_texts]
negative_examples = [{ 'text': text, 'label': 'negative' } for text in negative_texts]

# List comprehension is the same as doing this:
# positive_examples = []
# for text in positive_texts:
#     positive_examples.append({ 'text': text, 'label': 'positive' })


Combine these

In [ ]:
all_examples = positive_examples + negative_examples

**Important**! Shuffle your data so that you don't have all positives first followed by all negatives

In [ ]:
random.shuffle(all_examples)

Split into train, validation and test sets

In [ ]:
# Option 1: split completely manually
total_size = len(all_examples)

train_size = int(0.8*total_size)
valid_size = int(0.1*total_size)
test_size = total_size - train_size - valid_size

train_examples = all_examples[:train_size]
valid_examples = all_examples[train_size:train_size+valid_size]
test_examples = all_examples[train_size+valid_size:]

Let's see that our datasets are as we expected. They should be the correct sizes and have about the same proportion of negative and positive examples

In [ ]:
def check_data_integrity(train, valid, test):
  print('Examples in train, validation and test sets:')
  print(len(train), len(valid), len(test))
  print()
  print('Proportion of "positive" in the datasets:')
  print('Train:', len([example['label'] for example in train if example['label']=='positive' or example['label'] == 1])/len(train))
  print('Validation:', len([example['label'] for example in valid if example['label']=='positive' or example['label'] == 1])/len(valid))
  print('Test:',len([example['label'] for example in test if example['label']=='positive' or example['label'] == 1])/len(test))

In [ ]:
check_data_integrity(train_examples, valid_examples, test_examples)

In [ ]:
# Option 2: split using sklearn's train_test_split
# Here, we also stratify the data to ensure we get exactly
# 50/50 positive and negative examples in each split

from sklearn.model_selection import train_test_split

# Get a list of all labels for stratification
all_labels = [example['label'] for example in all_examples]

# First, split into train and test
train_examples, test_examples = train_test_split(all_examples,
                                                 train_size=0.8,
                                                 random_state=42,
                                                 stratify=all_labels)

# Then, split test in half to get test and validation
test_labels = [example['label'] for example in test_examples]

test_examples, valid_examples = train_test_split(test_examples,
                                                 train_size=0.5,
                                                 random_state=42,
                                                 stratify=test_labels)

In [ ]:
check_data_integrity(train_examples, valid_examples, test_examples)

Reformat as datasets. Right now, our data are lists of dictionaries, so we can use `Dataset.from_list` to convert them into datasets.

In [ ]:
train = Dataset.from_list(train_examples)
valid = Dataset.from_list(valid_examples)
test = Dataset.from_list(test_examples)

Create `DatasetDict` to wrap the `Dataset` objects

In [ ]:
data = {
    'train': train,
    'validation': valid,
    'test': test,
}

dataset = DatasetDict(data)

Check final dataset

In [ ]:
dataset

In [ ]:
dataset['train'][0]

## Alternative approach using Datasets early

In [ ]:
positive_texts = open('imdb-positives.txt').readlines()
negative_texts = open('imdb-negatives.txt').readlines()

positive_examples = [{ 'text': text, 'label': 'positive' } for text in positive_texts]
negative_examples = [{ 'text': text, 'label': 'negative' } for text in negative_texts]

all_examples = positive_examples + negative_examples

In [ ]:
# In order to use the 'stratify_by_label' argument, we need to assign features
features = datasets.Features({
      'text': datasets.Value('string'),
      'label': datasets.ClassLabel(names=['negative', 'positive'])
      })

ds = Dataset.from_list(all_examples, features)

In [ ]:
# Now the dataset has features
ds.features

In [ ]:
# And we can access the label names
ds.features['label'].names

In [ ]:
# NOTE! Now the class names have been mapped to 0 and 1
ds[0]

In [ ]:
# Now, we can split into train and test with stratification
# Unlike sklearns train_test_split, this method returns a DatasetDict
ds_train_test = ds.train_test_split(train_size=0.8, stratify_by_column="label")
ds_train_test

In [ ]:
# Split the test set into valid and test
ds_valid_test = ds_train_test['test'].train_test_split(test_size=0.5, stratify_by_column="label")
ds_valid_test

In [ ]:
data = datasets.DatasetDict({
    'train': ds_train_test['train'],
    'validation': ds_valid_test['train'],
    'test': ds_valid_test['test']
})
data

In [ ]:
check_data_integrity(data['train'], data['validation'], data['test'])